# Fine-tuning CroSloEngual BERT for Sentiment Classification

In this notebook, we will fine-tune [CroSloEngual BERT](https://arxiv.org/abs/2006.07890) — a multilingual encoder-only transformer model — for a three-class sentiment classification task on Slovenian text. We will apply **full fine-tuning**, updating all model parameters and replacing the model's original classification head with a new one matching our label set.

The notebook covers the complete training pipeline: loading the prepared dataset, tokenizing inputs, configuring training hyperparameters with the Hugging Face `Trainer` API, training the model, and saving the final checkpoint.

In [ ]:
!pip3 install transformers accelerate datasets evaluate scikit-learn

## Load Data

We will work with the dataset we prepared in the previous task. For easier loading, the processed version is already available on HuggingFace. We will simply download the data using `load_dataset` function. We will extract the training part and the validation part, that will serve as a "safeguard" to prevent overfitting.

In [1]:
from datasets import load_dataset

dataset = load_dataset("zivast/KKS-sentiment-workshop")
train_dataset = dataset["train"]
print("Train data size", len(train_dataset))
val_dataset = dataset["validation"]
print("Val data size", len(val_dataset))

train_dataset

/home/nikola/projects/studies/slaif/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating test split: 100%|██████████| 478/478 [00:00<00:00, 194970.08 examples/s]

Train data size 3821
Val data size 478


Dataset({
    features: ['text', 'label', 'label_text'],
    num_rows: 3821
})

## Load Model

In this task we will work with [CroSloEngual BERT](https://arxiv.org/abs/2006.07890), that was developed during the EMBEDDIA project. As the name suggests, the model was trained on a combination of Slovene, English and Croatian data. It serves as a good starting point for classification tasks in Slovene. Another encoder-type alternative for Slovene is [SloBERTa](https://huggingface.co/EMBEDDIA/sloberta).

CroSloEngual BERT is available on HuggingFace. We are going to load it using `AutoModelForSequenceClassification.from_pretrained` method, that loads the pretrained model and replaces its final layer with classification head for our task. For that purpose, we need to let the method now, how many classes (different labels) are in our dataset.

In [21]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_name = "EMBEDDIA/crosloengual-bert"
num_labels = len(set(train_dataset["label"]))
print("Number of labels:", num_labels)

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)
tokenizer = AutoTokenizer.from_pretrained(model_name)

Number of labels: 3


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 43927.50it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: EMBEDDIA/crosloengual-bert
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	

## Prepare data

The data preparation for the BERT models is simple. Since the models do not expect any prompt like decoder-type models, but only the input text, our only task is to tokenize the input. Additional constraint that we have to take into the account is that CroSloEngual BERT accepts the input sequences of maximum **512 tokens**. Therefore, we need to truncate potentially too long inputs to this length.

Such truncation can have a negative impact on the classification process - the key information for the classification could be hidden in the truncated part of the text (this is also true for the sentiment classification, where even the last input sentence can have a major impact). However, due to the model's limitation such step is necessary (modern decoder-type models with much longer context windows do not have this problem anymore).

We can perform the truncation and tokenization at the same time by applying the model's tokenizer to our input. When we call the tokenizer, we can set `truncation=True` and set `max_length` parameter accordingly (in our case to `512`). To tokenize every example in the dataset, we use `map` function that applies given function (`tokenize_example` in our case) to every example in the dataset. We can also parallelize its execution by setting the `num_proc` argument. In our case, we process 8 examples at once.

In [22]:
MAX_LENGTH = 512

def tokenize_example(example):
    return tokenizer(example["text"], truncation=True, max_length=MAX_LENGTH)

train_dataset = train_dataset.map(tokenize_example, num_proc=8)
val_dataset = val_dataset.map(tokenize_example, num_proc=8)
print("First training example:", train_dataset[0])

Map (num_proc=8): 100%|██████████| 478/478 [00:00<00:00, 880.66 examples/s]

First training example: {'text': 'Samo 30% igralcev lahko preseže mejo 100 metrov na smučarski skakalnici. Vašemu prijatelju je že uspelo. Lahko tudi vam? Pojdite skakat na: s1.skijumpmania.com/r50261', 'label': 1, 'label_text': 'neutral', 'input_ids': [103, 2274, 1203, 46852, 10460, 1107, 35900, 12003, 1434, 3361, 1008, 14520, 32485, 1120, 47105, 25176, 1427, 4942, 1015, 1001, 1104, 3184, 47105, 6317, 1039, 1477, 47144, 1256, 7109, 1679, 37487, 1064, 1008, 46811, 1022, 1246, 47105, 6124, 1081, 5060, 11472, 1002, 47105, 2318, 47302, 4497, 7249, 23194, 1246, 104], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}


## Setup Training

Before starting the training, we have to define our training parameters. In `transformers` library, this is handled by `TrainingArguments` class that represents the training config. Additionally, we can provide custom metrics, that are computed during the evaluation besides log-loss. In our case, we are interested in the accuracy of our classifier. Moreover, since the data is unbalanced (as we saw in the data preparation tasks, most of the examples have negative sentiment), we will also include F1-score as our metric of interest. To compute these metrics during the training, we define `compute_metrics`, that receives the logits (outputed probabilities for each label from the model) and ground truth labels and computes the metrics using existing Scikit-Learn functions.

In [23]:
import numpy as np
import evaluate

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=predictions, references=labels)["accuracy"],
        "f1": f1.compute(predictions=predictions, references=labels, average="macro")["f1"]
    }

In the `TrainingArguments` class we define parameters such as batch size, number of epochs, learning rate, optimizer, learning rate scheduler, etc. You can explore the hyperparameter values below and try to play around with them.

We will use the batch size of 16 for the demonstration. Since we are dealing with the BERT model, we can fit 16 examples on device and the micro batch size will be consequently the same as the global batch size (no gradient accumulation). We will use AdamW optimizer, that is typically used for the training of language models. Moreover, we will used fused version - it is the same as the standard one, but highly optimized for CUDA and therefore results in faster optimizer step. We will use cosine learning rate scheduler with a linear warmup, that is also standard in such training. We will train the model for 3 epochs.

The more interesting settings are save and eval steps. We set them in a way, that we perform evaluation twice per epoch and we save the model at the same step (this is good practice). Eval and save strategy can also be set to `epoch`, which saves/evaluates the checkpoint after each epoch. However, if often makes sense to save/evaluate more often. Saving after every N steps can prevent losing the progress when training larger models and the hardware or training fails for some reason and regular evaluation steps can prevent overfitting. To prevent saving an overfitted version of the model at the end we set `load_best_model_at_end=True`. As a control metric for the best model we define the evaluation loss and we tell the trainer, that we want to minimize it.

To monitor the progress of our training, we can export our metrics to external platform such as Weights & Biases, where we can observe various metrics through graphs. This is controled by the `report_to` parameter (which we could set to `wandb` or some other supported platform). However, in this demonstration we will not use any additional platform, but will print the metrics to the standard output. Therefore we set the `report_to` to `none`. Feel free to modify this value if you want to export your metrics to some platform.

If you want to know more about the other hyperparameters you can set, you can check the [TrainingArguments documentation](https://huggingface.co/docs/transformers/en/main_classes/trainer#transformers.TrainingArguments).

In [24]:
from transformers import TrainingArguments

# Batch size per GPU
micro_batch_size = 16
# Global batch size
batch_size = 16
# Accumulate the gradients to achieve the batch size
gradient_accumulation_steps = batch_size // micro_batch_size

# Number of training epochs
num_epochs = 3
steps_per_epoch = len(train_dataset) // batch_size
eval_steps = int(1 / 2 * steps_per_epoch)  # Evaluate 2 times per epoch
save_steps = int(1 / 2 * steps_per_epoch)  # Save 2 times per epoch

print(f"Steps per epoch: {steps_per_epoch}")
print(f"Evaluate each {eval_steps} steps ({eval_steps / steps_per_epoch:.2f} epochs)")
print(f"Save each {save_steps} steps ({save_steps / steps_per_epoch:.2f} epochs)")

args = TrainingArguments(
    # Output settings
    output_dir="tmp",  # Directory to save model checkpoints
    # Training duration
    num_train_epochs=num_epochs,
    # Batch size settings
    per_device_train_batch_size=micro_batch_size,
    per_device_eval_batch_size=micro_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    # Optimizer settings
    optim="adamw_torch_fused",  # Use fused AdamW for efficiency
    learning_rate=2e-5,  # Standard fine-tuning LR for BERT
    weight_decay=0.01,
    # Learning rate schedule
    warmup_steps=100,  # Number of steps when LR is linearly increasing
    lr_scheduler_type="cosine_with_min_lr",
    lr_scheduler_kwargs={"min_lr": 2e-6},
    # Logging and saving
    logging_steps=10,  # Log metrics every N steps
    eval_on_start=True, # Perform first evaluation before starting the training
    eval_strategy="steps",
    eval_steps=eval_steps,
    save_strategy="steps",
    save_steps=save_steps,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    # Precision settings
    bf16=True,
    # Integration settings
    push_to_hub=False,  # Don't push to HuggingFace Hub
    report_to="none"  # Disable external logging
)

Steps per epoch: 238
Evaluate each 119 steps (0.50 epochs)
Save each 119 steps (0.50 epochs)


## Train

To train the model we simply initialize the  `Trainer` object provided by the `transformers` library and call its `train` method. We need to pass the model (and the tokenizer), training and evaluation dataset and `TrainingArguments` to the trainer. We additionally pass the `compute_metrics` function that we prepared before and the `DataCollatorWithPadding`, which pads all the example in the same batch to the same length. This is useful for better GPU utilization and better training speed.

In [25]:
from transformers import Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=args,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

Step,Training Loss,Validation Loss,Accuracy,F1
0,No log,1.180987,0.238494,0.237923
119,0.709998,0.568686,0.763598,0.549029
238,0.664878,0.544269,0.778243,0.644290
357,0.361290,0.584614,0.769874,0.655107
476,0.377167,0.602458,0.759414,0.659206
595,0.207998,0.665105,0.757322,0.649543
714,0.266170,0.697416,0.748954,0.644273
717,0.266170,0.698216,0.748954,0.644273


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.79it/s]


TrainOutput(global_step=717, training_loss=0.41117664710248364, metrics={'train_runtime': 65.7094, 'train_samples_per_second': 174.45, 'train_steps_per_second': 10.912, 'total_flos': 1572402784666896.0, 'train_loss': 0.41117664710248364, 'epoch': 3.0})

## Save model

When the training is done, we can easily save the model by calling the trainer's `save_model` method that saves the model'S weights and configuration to the provided path. We need to save the tokenizer separately.

In [20]:
model_output_path = "mm-BERT-Sent"
trainer.save_model(model_output_path)
tokenizer.save_pretrained(model_output_path)

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.30s/it]


('mm-BERT-Sent/tokenizer_config.json', 'mm-BERT-Sent/tokenizer.json')